In [6]:
import kagglehub
kagglehub.login()

Kaggle credentials set.
Kaggle credentials successfully validated.


In [9]:
path = kagglehub.competition_download('jan-2026-dl-gen-ai-project')

print("Path to competition files:", path)

100%|██████████| 16.1G/16.1G [04:53<00:00, 58.9MB/s]

Extracting files...


Path to competition files: root/.cache/kagglehub/competitions/jan-2026-dl-gen-ai-project


In [10]:
import os
import torch
import numpy as np
import pandas as pd
import librosa
import random
from PIL import Image
from glob import glob
from tqdm import tqdm
from sklearn.metrics import accuracy_score, f1_score
from torch.utils.data import Dataset
from transformers import (
    ViTImageProcessor,
    ViTForImageClassification,
    TrainingArguments,
    Trainer,
    EarlyStoppingCallback,
)

# Configuration
MODEL_PATH = "vit-mashup-best-local"
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {DEVICE}")

# Load Model & Processor
checkpoint = "google/vit-base-patch16-224-in21k"
processor = ViTImageProcessor.from_pretrained(checkpoint)
model = ViTForImageClassification.from_pretrained(
    checkpoint,
    num_labels=10,
    ignore_mismatched_sizes=True
)
model.to(DEVICE)
print("Model Loaded.")

Using device: cuda


Loading weights:   0%|          | 0/198 [00:00<?, ?it/s]

ViTForImageClassification LOAD REPORT from: google/vit-base-patch16-224-in21k
Key                 | Status     | 
--------------------+------------+-
pooler.dense.weight | UNEXPECTED | 
pooler.dense.bias   | UNEXPECTED | 
classifier.bias     | MISSING    | 
classifier.weight   | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Model Loaded.


In [18]:
# Dataset Preset
dataset_path = os.path.expanduser("~/.cache/kagglehub/competitions/jan-2026-dl-gen-ai-project")
basePath = os.path.join(dataset_path, "messy_mashup")
genreStemPath = os.path.join(basePath, "genres_stems")
noisePath = os.path.join(basePath, "ESC-50-master", "audio")
WORK_DIR = os.path.join(os.getcwd(), "working")
os.makedirs(WORK_DIR, exist_ok=True)

genres = [
    'blues', 'classical', 'country', 'disco', 'hiphop', 'jazz', 'metal',
    'pop', 'reggae', 'rock'
]

stemKeysFile = ["drums.wav", "bass.wav", "vocals.wav", "other.wav"]
noise_files = glob(os.path.join(noisePath, "*.wav"))

SR = 16_000
CHUNK_SECONDS = 5
SAMPLES = SR * CHUNK_SECONDS

In [16]:
def to_mel(audio):
    mel = librosa.feature.melspectrogram(
        y=audio,
        sr=SR,
        n_fft=2048,
        hop_length=512,
        n_mels=128,
        fmin=20,
        fmax=SR // 2
    )
    mel = librosa.power_to_db(mel, ref=np.max)
    return mel.astype(np.float32)

def mel_to_image(mel):
    mel_norm = mel - mel.min()
    mel_max = mel_norm.max()
    if mel_max > 0:
        mel_norm = (mel_norm / mel_max * 255).astype(np.uint8)
    else:
        mel_norm = np.zeros_like(mel, dtype=np.uint8)
    return Image.fromarray(mel_norm).resize((224, 224), Image.BILINEAR).convert("RGB")

class MashupDataset(Dataset):
    def __init__(self, split_type="train"):
        self.data = {}
        for g in genres:
            p = os.path.join(genreStemPath, g)
            all_samples = sorted([s for s in os.listdir(p) if os.path.isdir(os.path.join(p, s))])
            split_idx = int(len(all_samples) * 0.8)
            if split_type == "train":
                self.data[g] = all_samples[:split_idx]
            else:
                self.data[g] = all_samples[split_idx:]

    def __len__(self):
        return sum(len(self.data[g]) for g in genres)

    def __getitem__(self, idx):
        genre = random.choice(genres)
        sample_id = random.choice(self.data[genre])

        stems = []
        for stem in stemKeysFile:
            s_path = os.path.join(genreStemPath, genre, sample_id, stem)
            audio, _ = librosa.load(s_path, sr=SR, duration=CHUNK_SECONDS)
            if len(audio) < SAMPLES: audio = np.pad(audio, (0, SAMPLES - len(audio)))
            stems.append(audio)

        # Mix and augment with noise
        mix = sum(stems) / len(stems)
        if len(noise_files) > 0:
            n_path = random.choice(noise_files)
            noise, _ = librosa.load(n_path, sr=SR, duration=CHUNK_SECONDS)
            if len(noise) < SAMPLES: noise = np.pad(noise, (0, SAMPLES - len(noise)))
            mix = mix + 0.005 * noise

        img = mel_to_image(to_mel(mix))
        pixel_values = processor(images=img, return_tensors="pt")["pixel_values"].squeeze(0)
        return {"pixel_values": pixel_values, "labels": torch.tensor(genres.index(genre))}

In [20]:
def compute_metrics(eval_pred):
    preds, labels = eval_pred
    preds = np.argmax(preds, axis=1)
    return {
        "accuracy": accuracy_score(labels, preds),
        "f1": f1_score(labels, preds, average="weighted")
    }

model.config.id2label = {i: g for i, g in enumerate(genres)}
model.config.label2id = {g: i for i, g in enumerate(genres)}

train_ds = MashupDataset(split_type="train")
val_ds = MashupDataset(split_type="val")

args = TrainingArguments(
    output_dir=MODEL_PATH,
    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=1,
    load_best_model_at_end=True,
    num_train_epochs=40,
    learning_rate=2e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    logging_steps=10,
    remove_unused_columns=False,
    report_to="none"
)

trainer = Trainer(
    model=model,
    args=args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=3)]
)

trainer.train()

Epoch,Training Loss,Validation Loss


KeyboardInterrupt: 

In [ ]:
def predict_genre(audio_path):
    audio, _ = librosa.load(audio_path, sr=SR, duration=CHUNK_SECONDS)
    if len(audio) < SAMPLES: audio = np.pad(audio, (0, SAMPLES - len(audio)))

    img = mel_to_image(to_mel(audio))
    inputs = processor(images=img, return_tensors="pt").to(DEVICE)

    model.eval()
    with torch.no_grad():
        logits = model(**inputs).logits

    predicted_id = logits.argmax(-1).item()
    return model.config.id2label[predicted_id]